# BC Harmony and Leiden Parameter Review

This notebook reproduces the BC 9-sample Harmony workflow and makes it easy to tune Leiden resolutions.

Use `RUN_HARMONY = False` if you only want to change Leiden resolutions on the existing 9-sample Harmony embedding. Use `RUN_HARMONY = True` if you want to change Harmony/PCA/neighbors/UMAP parameters and regenerate the corrected object.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/retina_mpl")
os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/retina_numba_cache")
os.environ.setdefault("PYTHONPYCACHEPREFIX", "/private/tmp/retina_pycache")

import harmonypy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse

sc.settings.verbosity = 2

# Avoid sc.settings.set_figure_params here: Scanpy 1.9.3 can call an
# IPython API that is missing in newer notebook/IPython versions.
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"

## Parameters

Edit this cell first. For ordinary cluster tuning, change only `LEIDEN_RESOLUTIONS` and keep `RUN_HARMONY = False`.

In [ ]:
ORIGINAL_BC_H5AD = Path("/Users/hoanglab/Desktop/vscode_projects/zebrafish-singlecell-portal/BC_annotated_clustered_corrected_doubletRemoved_Zebrafishes.h5ad")
EXISTING_9_SAMPLE_HARMONY_H5AD = Path("../results/h5ad/bc_strict_harmony_by_9_samples_reprocessed.h5ad")
OUTPUT_H5AD = Path("../results/h5ad/bc_notebook_harmony_reprocessed.h5ad")
FIGURES_DIR = Path("../results/figures/bc_notebook_leiden_review")
TABLES_DIR = Path("../results/tables/bc_notebook_leiden_review")

RUN_HARMONY = False  # False = load existing 9-sample Harmony h5ad and only redo Leiden.
SPLIT_ZEBRA_SAMPLE = True
BATCH_KEY = "sample_split_zebra"  # Used when RUN_HARMONY=True. If SPLIT_ZEBRA_SAMPLE=True, this is created automatically.

N_TOP_GENES = 3000
N_PCS = 50
N_NEIGHBORS = 15
RANDOM_STATE = 0

HARMONY_THETA = 3
HARMONY_MAX_ITER = 20
HARMONY_EPSILON_CLUSTER = 1e-6
HARMONY_EPSILON_HARMONY = 1e-5
HARMONY_SIGMA = 0.1
HARMONY_TAU = 0

LEIDEN_RESOLUTIONS = [1.0, 1.3, 1.5, 1.8, 2.0]
MARKER_GENES = ["nr2e3", "pde6a", "guca1b", "vsx1", "vsx2", "cabp5a", "cabp5b", "grm6a", "grm6b", "pcp4a"]

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)

## Helper Functions

In [ ]:
def res_tag(resolution):
    return str(resolution).replace(".", "p")


def sort_key(value):
    value = str(value)
    return int(value) if value.isdigit() else value


def add_sample_split_zebra(adata):
    sample_split = adata.obs["sample"].astype(str).copy()
    condition = adata.obs["renamed_samples"].astype(str)
    zebra_mask = sample_split == "Zebra"
    sample_split.loc[zebra_mask] = "Zebra_" + condition.loc[zebra_mask]
    adata.obs["sample_split_zebra"] = sample_split.astype("category")
    return adata


def add_cluster_labels(ax, coords, clusters, fontsize=9):
    clusters = clusters.astype(str)
    for cluster in sorted(clusters.unique(), key=sort_key):
        mask = clusters == cluster
        xy = coords[mask.to_numpy()]
        ax.text(
            float(np.median(xy[:, 0])),
            float(np.median(xy[:, 1])),
            cluster,
            ha="center",
            va="center",
            fontsize=fontsize,
            fontweight="bold",
            color="black",
            bbox={"boxstyle": "round,pad=0.16", "facecolor": "white", "edgecolor": "black", "linewidth": 0.4, "alpha": 0.78},
        )


def plot_cluster_umap(adata, cluster_key, title, out_prefix, label_fontsize=9):
    fig = sc.pl.umap(
        adata,
        color=cluster_key,
        legend_loc=None,
        show=False,
        return_fig=True,
        size=3,
        frameon=True,
    )
    fig.set_size_inches(11.5, 8.0)
    ax = fig.axes[0]
    add_cluster_labels(ax, adata.obsm["X_umap"], adata.obs[cluster_key], fontsize=label_fontsize)
    ax.set_title(title, fontsize=22)
    ax.set_xlabel("UMAP1", fontsize=16)
    ax.set_ylabel("UMAP2", fontsize=16)
    fig.savefig(FIGURES_DIR / f"{out_prefix}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{out_prefix}.pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_gene_umap_with_labels(adata, gene, cluster_key, out_prefix):
    fig = sc.pl.umap(
        adata,
        color=gene,
        use_raw=adata.raw is not None,
        cmap="viridis",
        show=False,
        return_fig=True,
        size=3,
        frameon=True,
    )
    fig.set_size_inches(14, 9.2)
    ax = fig.axes[0]
    add_cluster_labels(ax, adata.obsm["X_umap"], adata.obs[cluster_key], fontsize=13)
    ax.set_title(f"{gene} | {cluster_key}", fontsize=24)
    fig.savefig(FIGURES_DIR / f"{out_prefix}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{out_prefix}.pdf", dpi=600, bbox_inches="tight")
    plt.close(fig)


def get_expression_matrix(adata, genes):
    source = adata.raw if adata.raw is not None else adata
    lookup = {str(g).lower(): str(g) for g in source.var_names}
    matched = [lookup[g.lower()] for g in genes if g.lower() in lookup]
    X = source[:, matched].X
    if sparse.issparse(X):
        X = X.toarray()
    return np.asarray(X, dtype=float), matched

## Load or Recompute Harmony

If `RUN_HARMONY=False`, this loads the existing 9-sample Harmony h5ad. If `RUN_HARMONY=True`, it reruns the full correction from the original BC h5ad.

In [ ]:
if not RUN_HARMONY:
    adata = sc.read_h5ad(EXISTING_9_SAMPLE_HARMONY_H5AD)
    if SPLIT_ZEBRA_SAMPLE and "sample_split_zebra" not in adata.obs:
        adata = add_sample_split_zebra(adata)
    print(f"Loaded existing Harmony object: {adata.shape}")
else:
    adata = sc.read_h5ad(ORIGINAL_BC_H5AD)
    if SPLIT_ZEBRA_SAMPLE:
        adata = add_sample_split_zebra(adata)
        BATCH_KEY = "sample_split_zebra"
    if adata.raw is not None:
        adata.X = adata.raw.X.copy()

    sc.pp.filter_cells(adata, min_counts=1)
    if sparse.issparse(adata.X):
        adata.X.data = np.clip(adata.X.data, 0, None)
    else:
        adata.X = np.clip(adata.X, 0, None)
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    if sparse.issparse(adata.X):
        adata.X.data = np.nan_to_num(adata.X.data, nan=0, posinf=0, neginf=0).astype("float32", copy=False)
    else:
        adata.X = np.nan_to_num(adata.X, nan=0, posinf=0, neginf=0).astype("float32", copy=False)

    sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES, flavor="seurat")
    adata_hvg = adata[:, adata.var["highly_variable"].to_numpy()].copy()
    sc.pp.scale(adata_hvg, max_value=10)
    sc.tl.pca(adata_hvg, n_comps=N_PCS, svd_solver="arpack", random_state=RANDOM_STATE)

    harmony_out = harmonypy.run_harmony(
        adata_hvg.obsm["X_pca"],
        adata_hvg.obs,
        BATCH_KEY,
        theta=HARMONY_THETA,
        max_iter_harmony=HARMONY_MAX_ITER,
        epsilon_cluster=HARMONY_EPSILON_CLUSTER,
        epsilon_harmony=HARMONY_EPSILON_HARMONY,
        sigma=HARMONY_SIGMA,
        tau=HARMONY_TAU,
    )
    corrected = harmony_out.Z_corr
    if corrected.shape[0] != adata.n_obs and corrected.shape[1] == adata.n_obs:
        corrected = corrected.T

    adata.obsm["X_pca_uncorrected_strict"] = adata_hvg.obsm["X_pca"].astype("float32")
    adata.obsm["X_pca_harmony_strict"] = corrected.astype("float32")
    adata.obsm["X_pca_harmony"] = adata.obsm["X_pca_harmony_strict"]
    adata.obsm["X_pca"] = adata.obsm["X_pca_harmony_strict"]
    sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS, use_rep="X_pca_harmony_strict", random_state=RANDOM_STATE)
    sc.tl.umap(adata, random_state=RANDOM_STATE)
    adata.write_h5ad(OUTPUT_H5AD, compression="gzip")
    print(f"Wrote {OUTPUT_H5AD}")

adata

## Check Batch Labels and Baseline UMAP

In [ ]:
display(pd.crosstab(adata.obs["sample"].astype(str), adata.obs["renamed_samples"].astype(str)))
if "sample_split_zebra" in adata.obs:
    display(adata.obs["sample_split_zebra"].astype(str).value_counts().rename_axis("sample_split_zebra").reset_index(name="n_cells"))

sc.pl.umap(adata, color="sample_split_zebra" if "sample_split_zebra" in adata.obs else "sample", size=3, frameon=True)

## Run Leiden at Multiple Resolutions

In [ ]:
resolution_summary = []
assignments = pd.DataFrame(index=adata.obs_names)

for resolution in LEIDEN_RESOLUTIONS:
    tag = res_tag(resolution)
    key = f"leiden_res_{tag}"
    sc.tl.leiden(adata, resolution=resolution, key_added=key, random_state=RANDOM_STATE)
    counts = (
        adata.obs[key]
        .astype(str)
        .value_counts()
        .rename_axis(key)
        .reset_index(name="n_cells")
        .sort_values(key, key=lambda s: s.map(sort_key))
    )
    counts.to_csv(TABLES_DIR / f"counts_by_{key}.csv", index=False)
    assignments[key] = adata.obs[key].astype(str)
    resolution_summary.append({
        "resolution": resolution,
        "cluster_key": key,
        "n_clusters": int(adata.obs[key].nunique()),
        "min_cluster_size": int(counts["n_cells"].min()),
        "median_cluster_size": float(counts["n_cells"].median()),
        "max_cluster_size": int(counts["n_cells"].max()),
    })
    plot_cluster_umap(adata, key, f"BC 9-sample Harmony | Leiden {resolution}", f"bc_notebook_{key}_cluster_numbers")

resolution_summary = pd.DataFrame(resolution_summary)
resolution_summary.to_csv(TABLES_DIR / "leiden_resolution_summary.csv", index=False)
assignments.to_csv(TABLES_DIR / "leiden_resolution_assignments.csv")
display(resolution_summary)

## Inspect One Resolution

In [ ]:
SELECTED_RESOLUTION = 2.0
SELECTED_CLUSTER_KEY = f"leiden_res_{res_tag(SELECTED_RESOLUTION)}"

sc.pl.umap(adata, color=SELECTED_CLUSTER_KEY, legend_loc="on data", size=3, frameon=True)
display(adata.obs[SELECTED_CLUSTER_KEY].astype(str).value_counts().sort_index(key=lambda s: s.map(sort_key)).rename_axis(SELECTED_CLUSTER_KEY).reset_index(name="n_cells"))

## Gene UMAPs with Cluster Labels

Use this for candidate contaminating/noisy clusters. These save high-resolution PNG/PDF files.

In [ ]:
GENES_TO_LABEL = ["nr2e3", "pde6a", "guca1b"]

for gene in GENES_TO_LABEL:
    plot_gene_umap_with_labels(
        adata,
        gene,
        SELECTED_CLUSTER_KEY,
        f"bc_notebook_umap_gene_{gene}_{SELECTED_CLUSTER_KEY}_labels_highres",
    )

print(f"Saved labeled gene UMAPs to {FIGURES_DIR}")

## Cluster-Level Marker Summary

This compares photoreceptor-like markers with BC markers per cluster. It is a helper view, not an automatic removal rule.

In [ ]:
photo_genes = ["nr2e3", "pde6a", "guca1b"]
bc_genes = ["vsx1", "vsx2", "cabp5a", "cabp5b", "grm6a", "grm6b", "pcp4a"]
all_marker_genes = photo_genes + bc_genes

X, matched_genes = get_expression_matrix(adata, all_marker_genes)
clusters = adata.obs[SELECTED_CLUSTER_KEY].astype(str).to_numpy()

records = []
for cluster in sorted(pd.unique(clusters), key=sort_key):
    mask = clusters == cluster
    Xm = X[mask, :]
    for i, gene in enumerate(matched_genes):
        expr = Xm[:, i]
        records.append({
            "cluster": cluster,
            "gene": gene,
            "n_cells": int(mask.sum()),
            "mean_expression": float(np.mean(expr)),
            "fraction_expressing": float(np.mean(expr > 0)),
        })

summary = pd.DataFrame(records)
summary.to_csv(TABLES_DIR / f"marker_summary_{SELECTED_CLUSTER_KEY}.csv", index=False)

mean = summary.pivot(index="cluster", columns="gene", values="mean_expression").fillna(0)
mean = mean.loc[sorted(mean.index, key=sort_key), matched_genes]

fig, ax = plt.subplots(figsize=(10.5, max(7, 0.22 * len(mean) + 2.5)))
sns.heatmap(mean, cmap="viridis", linewidths=0.2, linecolor="white", ax=ax)
ax.set_title(f"Mean marker expression | {SELECTED_CLUSTER_KEY}")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"marker_mean_expression_heatmap_{SELECTED_CLUSTER_KEY}.png", dpi=300)
plt.show()

## Save Modified h5ad with New Cluster Columns

This is optional. It preserves the original embedding and adds all `leiden_res_*` columns.

In [ ]:
SAVE_CLUSTERED_H5AD = False
CLUSTERED_OUTPUT_H5AD = Path("../results/h5ad/bc_9_sample_harmony_with_notebook_leiden_resolutions.h5ad")

if SAVE_CLUSTERED_H5AD:
    adata.write_h5ad(CLUSTERED_OUTPUT_H5AD, compression="gzip")
    print(f"Wrote {CLUSTERED_OUTPUT_H5AD}")